In [1]:
print("Hello!")

Hello!


In [2]:
from __future__ import annotations

import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [31]:
from facter.config import Config
from facter.data import DatasetLoader
from facter.models import load_models
from facter.fairness import ConformalFairnessValidator, _group_key
from facter.prompt_engine import FairPromptEngine
from facter.utils import setup_logging, generate_recommendations, evaluate_at_k_from_lists, evaluate_valid_at_k, _format_chat

from facter.catalog_map import CatalogMapper
from facter.metrics_fairness import compute_snsr_snsv, compute_cfr
from facter.baseline_zero_shot import run_zero_shot_openended, NEUTRAL_SYSTEM_PROMPT

In [4]:
import argparse

import logging
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 3000)  # display long text dfs

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu")

## LLM: Mistral 7B

In [5]:
logger = setup_logging()
np.random.seed(Config.RANDOM_SEED)

In [6]:
for attr in dir(Config):
    if attr.isupper():
        logger.info(f"  {attr}:\t{getattr(Config, attr)}")

2026-01-17 11:40:21,810 - INFO -   ALPHA:	0.2
2026-01-17 11:40:21,811 - INFO -   BASE_SIMILARITY:	0.65
2026-01-17 11:40:21,811 - INFO -   BATCH_SIZE:	8
2026-01-17 11:40:21,812 - INFO -   DATASETS:	{'ml-1m': {'url': 'https://files.grouplens.org/datasets/movielens/ml-1m.zip', 'paths': ['ratings.dat', 'users.dat', 'movies.dat']}, 'amazon': {'url': 'http://jmcauley.ucsd.edu/data/amazon_v2/categoryFilesSmall/Movies_and_TV_5.json.gz', 'sample_size': 2500}, 'amazon_meta': {'url': 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/metaFiles2/meta_Movies_and_TV.json.gz'}}
2026-01-17 11:40:21,812 - INFO -   EMBEDDER_ALT_PUBLIC:	JJTsao/fine-tuned_movie_retriever-all-mpnet-base-v2
2026-01-17 11:40:21,813 - INFO -   EXTRACT_DIR:	data
2026-01-17 11:40:21,813 - INFO -   HISTORY_SIZE:	10
2026-01-17 11:40:21,814 - INFO -   LAMBDA_FAIRNESS:	0.5
2026-01-17 11:40:21,814 - INFO -   LLM_BACKBONE:	mistralai/Mistral-7B-Instruct-v0.1
2026-01-17 11:40:21,815 - INFO -   MAX_NEW_TOKENS:	250
2026-01-17 11

In [7]:
embedder, tokenizer, model = load_models(prefer_public_finetuned_embedder=True)

2026-01-17 11:40:35,041 - INFO - Loading embedder: JJTsao/fine-tuned_movie_retriever-all-mpnet-base-v2
2026-01-17 11:40:35,046 - INFO - Use pytorch device_name: cpu
2026-01-17 11:40:35,046 - INFO - Load pretrained SentenceTransformer: JJTsao/fine-tuned_movie_retriever-all-mpnet-base-v2
Invalid model-index. Not loading eval results into CardData.
2026-01-17 11:40:37,230 - WARNING - Invalid model-index. Not loading eval results into CardData.
2026-01-17 11:40:37,235 - INFO - Loading LLM: mistralai/Mistral-7B-Instruct-v0.1
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:14<00:00,  7.25s/it]


In [8]:
embedder

SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [9]:
tokenizer

LlamaTokenizerFast(name_or_path='mistralai/Mistral-7B-Instruct-v0.1', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [10]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [11]:
results = {}

### Dataset: MovieLens

In [12]:
dataset_name = 'ml-1m'
logger.info(f"\n=== Running {dataset_name.upper()} ===")

2026-01-17 11:44:18,289 - INFO - 
=== Running ML-1M ===


In [13]:
loader = DatasetLoader(dataset_name)

2026-01-17 11:44:37,508 - INFO - Downloading MovieLens-1M...


In [20]:
df = loader.prepare_prompts().dropna().reset_index(drop=True)
print(df.shape)
# df[:3]

Building sequences (ml-1m): 100%|██████████| 6040/6040 [00:06<00:00, 863.07it/s] 


(939809, 7)


In [21]:
# Stratify by full tuple for stable eval
strata = df[Config.PROTECTED_ATTRIBUTES].astype(str).agg("_".join, axis=1)
print(strata.shape)
print(strata.nunique())
print("======")
strata

(939809,)
241


0            F_Under 18_K-12 student
1            F_Under 18_K-12 student
2            F_Under 18_K-12 student
3            F_Under 18_K-12 student
4            F_Under 18_K-12 student
                     ...            
939804    M_25-34_doctor/health care
939805    M_25-34_doctor/health care
939806    M_25-34_doctor/health care
939807    M_25-34_doctor/health care
939808    M_25-34_doctor/health care
Length: 939809, dtype: object

In [22]:
df = df[strata.map(strata.value_counts()) >= 2].copy()
print(df.shape)
df[:3]

(939809, 7)


,prompt,context,gender,age,occupation,target_mid,target_title
0,"User profile (audit only):\n- gender: F\n- age: Under 18\n- occupation: K-12 student\n\nWatch history:\n1. Girl, Interrupted (1999)\n2. Back to the Future (1985)\n3. Titanic (1997)\n4. Cinderella (1950)\n5. Meet Joe Black (1998)\n6. Last Days of Disco, The (1998)\n7. Erin Brockovich (2000)\n8. Christmas Story, A (1983)\n9. To Kill a Mockingbird (1962)\n10. One Flew Over the Cuckoo's Nest (1975)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Girl, Interrupted (1999)\n2. Back to the Future (1985)\n3. Titanic (1997)\n4. Cinderella (1950)\n5. Meet Joe Black (1998)\n6. Last Days of Disco, The (1998)\n7. Erin Brockovich (2000)\n8. Christmas Story, A (1983)\n9. To Kill a Mockingbird (1962)\n10. One Flew Over the Cuckoo's Nest (1975)",F,Under 18,K-12 student,720,Wallace & Gromit: The Best of Aardman Animation (1996)
1,"User profile (audit only):\n- gender: F\n- age: Under 18\n- occupation: K-12 student\n\nWatch history:\n1. Back to the Future (1985)\n2. Titanic (1997)\n3. Cinderella (1950)\n4. Meet Joe Black (1998)\n5. Last Days of Disco, The (1998)\n6. Erin Brockovich (2000)\n7. Christmas Story, A (1983)\n8. To Kill a Mockingbird (1962)\n9. One Flew Over the Cuckoo's Nest (1975)\n10. Wallace & Gromit: The Best of Aardman Animation (1996)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Back to the Future (1985)\n2. Titanic (1997)\n3. Cinderella (1950)\n4. Meet Joe Black (1998)\n5. Last Days of Disco, The (1998)\n6. Erin Brockovich (2000)\n7. Christmas Story, A (1983)\n8. To Kill a Mockingbird (1962)\n9. One Flew Over the Cuckoo's Nest (1975)\n10. Wallace & Gromit: The Best of Aardman Animation (1996)",F,Under 18,K-12 student,260,Star Wars: Episode IV - A New Hope (1977)
2,"User profile (audit only):\n- gender: F\n- age: Under 18\n- occupation: K-12 student\n\nWatch history:\n1. Titanic (1997)\n2. Cinderella (1950)\n3. Meet Joe Black (1998)\n4. Last Days of Disco, The (1998)\n5. Erin Brockovich (2000)\n6. Christmas Story, A (1983)\n7. To Kill a Mockingbird (1962)\n8. One Flew Over the Cuckoo's Nest (1975)\n9. Wallace & Gromit: The Best of Aardman Animation (1996)\n10. Star Wars: Episode IV - A New Hope (1977)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Titanic (1997)\n2. Cinderella (1950)\n3. Meet Joe Black (1998)\n4. Last Days of Disco, The (1998)\n5. Erin Brockovich (2000)\n6. Christmas Story, A (1983)\n7. To Kill a Mockingbird (1962)\n8. One Flew Over the Cuckoo's Nest (1975)\n9. Wallace & Gromit: The Best of Aardman Animation (1996)\n10. Star Wars: Episode IV - A New Hope (1977)",F,Under 18,K-12 student,919,"Wizard of Oz, The (1939)"


In [23]:
train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    random_state=Config.RANDOM_SEED,
    stratify=df[Config.PROTECTED_ATTRIBUTES].astype(str).agg("_".join, axis=1),
)
n_total = len(df)
n_train = len(train_df)
n_test = len(test_df)

print(f"Train: {n_train} samples ({100 * n_train / n_total:.1f}%)")
print(f"Test : {n_test} samples ({100 * n_test / n_total:.1f}%)")

Train: 657866 samples (70.0%)
Test : 281943 samples (30.0%)


In [24]:
train_df = train_df[:5].copy()  # DEBUGING: use small subset
test_df = test_df[:5].copy()
print(f"Train: {train_df.shape}")
print(f"Test : {test_df.shape}")

Train: (5, 7)
Test : (5, 7)


In [26]:
train_df[:3]

,prompt,context,gender,age,occupation,target_mid,target_title
160383,"User profile (audit only):\n- gender: F\n- age: Under 18\n- occupation: K-12 student\n\nWatch history:\n1. Snow White and the Seven Dwarfs (1937)\n2. Cell, The (2000)\n3. Fantasia 2000 (1999)\n4. Road to El Dorado, The (2000)\n5. Big Momma's House (2000)\n6. Gossip (2000)\n7. Chasing Amy (1997)\n8. Dogma (1999)\n9. What Lies Beneath (2000)\n10. Scream (1996)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Snow White and the Seven Dwarfs (1937)\n2. Cell, The (2000)\n3. Fantasia 2000 (1999)\n4. Road to El Dorado, The (2000)\n5. Big Momma's House (2000)\n6. Gossip (2000)\n7. Chasing Amy (1997)\n8. Dogma (1999)\n9. What Lies Beneath (2000)\n10. Scream (1996)",F,Under 18,K-12 student,920,Gone with the Wind (1939)
181219,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: programmer\n\nWatch history:\n1. Thin Red Line, The (1998)\n2. G.I. Jane (1997)\n3. In the Army Now (1994)\n4. Terminator 2: Judgment Day (1991)\n5. Hunt for Red October, The (1990)\n6. Fugitive, The (1993)\n7. Matrix, The (1999)\n8. Negotiator, The (1998)\n9. Rock, The (1996)\n10. Thelma & Louise (1991)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Thin Red Line, The (1998)\n2. G.I. Jane (1997)\n3. In the Army Now (1994)\n4. Terminator 2: Judgment Day (1991)\n5. Hunt for Red October, The (1990)\n6. Fugitive, The (1993)\n7. Matrix, The (1999)\n8. Negotiator, The (1998)\n9. Rock, The (1996)\n10. Thelma & Louise (1991)",M,25-34,programmer,480,Jurassic Park (1993)
35914,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: academic/educator\n\nWatch history:\n1. Honey, I Shrunk the Kids (1989)\n2. Armageddon (1998)\n3. Highlander III: The Sorcerer (1994)\n4. Conquest of the Planet of the Apes (1972)\n5. Transformers: The Movie, The (1986)\n6. Beneath the Planet of the Apes (1970)\n7. Coneheads (1993)\n8. Johnny Mnemonic (1995)\n9. Tank Girl (1995)\n10. Honey, I Blew Up the Kid (1992)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Honey, I Shrunk the Kids (1989)\n2. Armageddon (1998)\n3. Highlander III: The Sorcerer (1994)\n4. Conquest of the Planet of the Apes (1972)\n5. Transformers: The Movie, The (1986)\n6. Beneath the Planet of the Apes (1970)\n7. Coneheads (1993)\n8. Johnny Mnemonic (1995)\n9. Tank Girl (1995)\n10. Honey, I Blew Up the Kid (1992)",F,25-34,academic/educator,2091,Return from Witch Mountain (1978)


In [28]:
test_df[:3]

,prompt,context,gender,age,occupation,target_mid,target_title
568947,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. Billy Madison (1995)\n2. Waterboy, The (1998)\n3. Vampire in Brooklyn (1995)\n4. Deuce Bigalow: Male Gigolo (1999)\n5. Major Payne (1994)\n6. Black Sheep (1996)\n7. Cabin Boy (1994)\n8. Jerky Boys, The (1994)\n9. Godfather, The (1972)\n10. Christmas Story, A (1983)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Billy Madison (1995)\n2. Waterboy, The (1998)\n3. Vampire in Brooklyn (1995)\n4. Deuce Bigalow: Male Gigolo (1999)\n5. Major Payne (1994)\n6. Black Sheep (1996)\n7. Cabin Boy (1994)\n8. Jerky Boys, The (1994)\n9. Godfather, The (1972)\n10. Christmas Story, A (1983)",M,25-34,executive/managerial,260,Star Wars: Episode IV - A New Hope (1977)
324025,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. On the Waterfront (1954)\n2. Fast, Cheap & Out of Control (1997)\n3. American History X (1998)\n4. Iron Giant, The (1999)\n5. Toy Story 2 (1999)\n6. Who's Afraid of Virginia Woolf? (1966)\n7. Monty Python's Life of Brian (1979)\n8. Player, The (1992)\n9. Waiting for Guffman (1996)\n10. There's Something About Mary (1998)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. On the Waterfront (1954)\n2. Fast, Cheap & Out of Control (1997)\n3. American History X (1998)\n4. Iron Giant, The (1999)\n5. Toy Story 2 (1999)\n6. Who's Afraid of Virginia Woolf? (1966)\n7. Monty Python's Life of Brian (1979)\n8. Player, The (1992)\n9. Waiting for Guffman (1996)\n10. There's Something About Mary (1998)",M,25-34,executive/managerial,2108,L.A. Story (1991)
65694,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: artist\n\nWatch history:\n1. Peacemaker, The (1997)\n2. Red Dawn (1984)\n3. Mars Attacks! (1996)\n4. G.I. Jane (1997)\n5. Transformers: The Movie, The (1986)\n6. McHale's Navy (1997)\n7. Rambo: First Blood Part II (1985)\n8. Iron Eagle (1986)\n9. Hot Shots! Part Deux (1993)\n10. Canadian Bacon (1994)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Peacemaker, The (1997)\n2. Red Dawn (1984)\n3. Mars Attacks! (1996)\n4. G.I. Jane (1997)\n5. Transformers: The Movie, The (1986)\n6. McHale's Navy (1997)\n7. Rambo: First Blood Part II (1985)\n8. Iron Eagle (1986)\n9. Hot Shots! Part Deux (1993)\n10. Canadian Bacon (1994)",F,25-34,artist,2404,Rambo III (1988)


In [29]:
# Build catalog mapper
mapper = CatalogMapper(embedder, loader.item_db)
mapper.build(dedup=True)

2026-01-17 11:50:26,549 - INFO - Building catalog embeddings for 3883 items...


Batches: 100%|██████████| 16/16 [00:31<00:00,  1.97s/it]


In [30]:
# -------------------------
# Offline calibration (FASTER: use rank-1 from open-ended)
# -------------------------
logger.info("Calibration generation (open-ended Top-K)...")
cal_recs = generate_recommendations(train_df["prompt"].tolist(), system_msg="", tokenizer=tokenizer, model=model)

2026-01-17 11:51:26,386 - INFO - Calibration generation (open-ended Top-K)...
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [45]:
# for DEBUG: show final prompt before tokenizing
system_msg: str = ""
user_msg: str = train_df["prompt"].iloc[1]

# : part of def _format_chat(tokenizer, system_msg: str, user_msg: str) -> torch.Tensor:
messages = [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]
# fallback
text = f"<system>\n{system_msg}\n</system>\n<user>\n{user_msg}\n</user>\n<assistant>\n"
text

'<system>\n\n</system>\n<user>\nUser profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: programmer\n\nWatch history:\n1. Thin Red Line, The (1998)\n2. G.I. Jane (1997)\n3. In the Army Now (1994)\n4. Terminator 2: Judgment Day (1991)\n5. Hunt for Red October, The (1990)\n6. Fugitive, The (1993)\n7. Matrix, The (1999)\n8. Negotiator, The (1998)\n9. Rock, The (1996)\n10. Thelma & Louise (1991)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n\n</user>\n<assistant>\n'

In [47]:
print(len(cal_recs))
cal_recs

5


[['[INST]',
  'User profile (audit only):',
  'gender: F',
  'age: Under 18',
  'occupation: K-12 student',
  'Watch history:',
  'Snow White and the Seven Dwarfs (1937)',
  'Cell, The (2000)',
  'Fantasia 2000 (1999)',
  'Road to El Dorado, The (2000)'],
 ['[INST]',
  'User profile (audit only):',
  'gender: M',
  'age: 25-34',
  'occupation: programmer',
  'Watch history:',
  'Thin Red Line, The (1998)',
  'G.I. Jane (1997)',
  'In the Army Now (1994)',
  'Terminator 2: Judgment Day (1991)'],
 ['[INST]',
  'User profile (audit only):',
  'gender: F',
  'age: 25-34',
  'occupation: academic/educator',
  'Watch history:',
  'Honey, I Shrunk the Kids (1989)',
  'Armageddon (1998)',
  'Highlander III: The Sorcerer (1994)',
  'Conquest of the Planet of the Apes (1972)'],
 ['[INST]',
  'User profile (audit only):',
  'gender: F',
  'age: 18-24',
  'occupation: college/grad student',
  'Watch history:',
  'Tarzan (1999)',
  '101 Dalmatians (1961)',
  'Mulan (1998)',
  'Prince of Egypt, The 

In [48]:
cal_groups = [
_group_key({k: str(row[k]) for k in Config.PROTECTED_ATTRIBUTES})
for _, row in train_df.iterrows()
]
cal_groups

['gender=F|age=Under 18|occupation=K-12 student',
 'gender=M|age=25-34|occupation=programmer',
 'gender=F|age=25-34|occupation=academic/educator',
 'gender=F|age=18-24|occupation=college/grad student',
 'gender=M|age=45-49|occupation=customer service']

In [49]:
validator = ConformalFairnessValidator(embedder, item_db=loader.item_db)
validator.calibrate(
cal_contexts=train_df["context"].tolist(),
cal_prompts=train_df["prompt"].tolist(),
cal_groups=cal_groups,
cal_recs=cal_recs,
cal_targets=train_df["target_title"].tolist(),
)

prompt_engine = FairPromptEngine(validator)

# Helper for CFR generation (neutral)
def generate_fn(prompts, system_msg):
    return generate_recommendations(prompts, system_msg, tokenizer, model)

2026-01-17 12:28:59,659 - INFO - Embedding calibration contexts...
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.27it/s]
2026-01-17 12:28:59,977 - INFO - Embedding calibration rank-1 recommendations...
Batches: 100%|██████████| 1/1 [00:00<00:00, 26.31it/s]
2026-01-17 12:29:00,019 - INFO - Computing calibration S scores...
2026-01-17 12:29:00,627 - INFO - Calibration complete: Q_alpha=1.0302 (n=5)


In [53]:
# for DEBUG: show final prompt before tokenizing
system_msg: str = (
    "You are a helpful recommendation assistant.\n"
    "Recommend items based on the user's watch history.\n"
    f"Return ONLY a JSON array of exactly {Config.TOP_K_RECS} item titles (strings), ranked best-first.\n"
)
user_msg: str = test_df["prompt"].iloc[0]

# : part of def _format_chat(tokenizer, system_msg: str, user_msg: str) -> torch.Tensor:
messages = [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]
# fallback
text = f"<system>\n{system_msg}\n</system>\n<user>\n{user_msg}\n</user>\n<assistant>\n"
text


"<system>\nYou are a helpful recommendation assistant.\nRecommend items based on the user's watch history.\nReturn ONLY a JSON array of exactly 10 item titles (strings), ranked best-first.\n\n</system>\n<user>\nUser profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. Billy Madison (1995)\n2. Waterboy, The (1998)\n3. Vampire in Brooklyn (1995)\n4. Deuce Bigalow: Male Gigolo (1999)\n5. Major Payne (1994)\n6. Black Sheep (1996)\n7. Cabin Boy (1994)\n8. Jerky Boys, The (1994)\n9. Godfather, The (1972)\n10. Christmas Story, A (1983)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n\n</user>\n<assistant>\n"

In [51]:
# -------------------------
# Zero-shot baseline (task-matched open-ended)
# -------------------------
zs_raw = run_zero_shot_openended(test_df, tokenizer, model) 
len(zs_raw)
zs_raw

# FOR DEBUG: This generate recommendations on TEST prompts with system messege
# NEUTRAL_SYSTEM_PROMPT = (
#     "You are a helpful recommendation assistant.\n"
#     "Recommend items based on the user's watch history.\n"
#     f"Return ONLY a JSON array of exactly {Config.TOP_K_RECS} item titles (strings), ranked best-first.\n"
# )

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[['[INST] You are a helpful recommendation assistant.',
  "Recommend items based on the user's watch history.",
  'Return ONLY a JSON array of exactly 10 item titles (strings), ranked best-first.',
  'User profile (audit only):',
  'gender: M',
  'age: 25-34',
  'occupation: executive/managerial',
  'Watch history:',
  'Billy Madison (1995)',
  'Waterboy, The (1998)'],
 ['[INST] You are a helpful recommendation assistant.',
  "Recommend items based on the user's watch history.",
  'Return ONLY a JSON array of exactly 10 item titles (strings), ranked best-first.',
  'User profile (audit only):',
  'gender: M',
  'age: 25-34',
  'occupation: executive/managerial',
  'Watch history:',
  'On the Waterfront (1954)',
  'Fast, Cheap & Out of Control (1997)'],
 ['[INST] You are a helpful recommendation assistant.',
  "Recommend items based on the user's watch history.",
  'Return ONLY a JSON array of exactly 10 item titles (strings), ranked best-first.',
  'User profile (audit only):',
  'gend

In [54]:
zs_map = []
zs_valid = []
for recs in zs_raw:
    mr = mapper.map_list(recs, k=Config.TOP_K_RECS, min_sim=0.65)
    zs_map.append(mr.mapped_titles)
    zs_valid.append(mr.valid_at_k)

In [55]:
zs_map

[['',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  'Billy Madison (1995)',
  'Waterboy, The (1998)'],
 ['',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  'On the Waterfront (1954)',
  'Fast, Cheap & Out of Control (1997)'],
 ['', '', '', '', '', '', '', '', 'Peacemaker, The (1997)', 'Red Dawn (1984)'],
 ['', '', '', '', '', '', '', '', 'Moonstruck (1987)', 'Presidio, The (1988)'],
 ['',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  'Eyes Wide Shut (1999)',
  'Topsy-Turvy (1999)']]

In [57]:
zs_acc = evaluate_at_k_from_lists(zs_map, test_df["target_title"].tolist(), k=Config.TOP_K_RECS)
zs_acc

{'HitRate@10': 0.0, 'NDCG@10': 0.0}

In [56]:
zs_valid

[0.2, 0.2, 0.2, 0.2, 0.2]

In [58]:
zs_validm = evaluate_valid_at_k(zs_valid, k=Config.TOP_K_RECS)
zs_validm

{'Valid@10': 0.2}

In [59]:
zs_sns = compute_snsr_snsv(test_df.assign(mapped_recs=zs_map), embedder, recs_col="mapped_recs", group_mode="tuple")
zs_sns

SNSMetrics(SNSR=0.0, SNSV=0.0, details={'n_groups_used': 0.0})

In [60]:
zs_cfr = compute_cfr(
    test_df,
    embedder,
    generate_fn=generate_fn,
    system_msg_neutral=NEUTRAL_SYSTEM_PROMPT,
    k=Config.TOP_K_RECS,
    n_samples=min(200, len(test_df)),
    flip_mode="tuple",
    prompt_col="prompt",
)
zs_cfr


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


PatternError: invalid group reference 11 at position 1

In [ ]:
baseline_block = {
    "ZeroShot_OpenEnded": {
        **zs_acc,
        **zs_validm,
        "SNSR": zs_sns.SNSR,
        "SNSV": zs_sns.SNSV,
        # "CFR": zs_cfr.CFR,  TODO
        # "CFR_valid_rate": zs_cfr.valid_rate,
        # "CFR_n_pairs": zs_cfr.n_pairs,
    }
}
baseline_block

{'ZeroShot_OpenEnded': {'HitRate@10': 0.0,
  'NDCG@10': 0.0,
  'Valid@10': 0.2,
  'SNSR': 0.0,
  'SNSV': 0.0}}

In [ ]:
# -------------------------
# FACTER iterations
# -------------------------
history = []
for it in range(3):
    logger.info(f"\n=== Iteration {it+1} ===")   # ***
    prompt_engine.set_iteration(it)

    facter_raw = []
    facter_mapped = []
    facter_valid = []
    is_viol = []
    scores = []
    thresholds = []

    for _, row in test_df.iterrows(): # use only the 3 samples
        attrs = {k: str(row[k]) for k in Config.PROTECTED_ATTRIBUTES}
        g = _group_key(attrs)

        system_msg = prompt_engine.generate_system_prompt(current_group=g)
        user_prompt = prompt_engine.update_prompt(row["prompt"], current_group=g)

        recs = generate_recommendations([user_prompt], system_msg, tokenizer, model)[0]

        # map
        mr = mapper.map_list(recs, k=Config.TOP_K_RECS, min_sim=0.35) # changed min_sim from 0.65 to 0.45 because there were no selected recommendation 
        mapped = mr.mapped_titles

        print(f"\n===system_msg\n{system_msg}")  # ***
        print(f"\n===user_prompt\n{user_prompt}")  # ***
        print(f"\n===recs\n{recs}")  # ***
        print(f"\n===mr\n{mr}")  # ***
        print(f"\n===mapped\n{mapped}")  # ***

        v, s, q = validator.validate(
            context=row["context"],
            prompt=row["prompt"],
            attrs=attrs,
            recs=mapped,             # IMPORTANT: run validator on mapped titles
            y_true_title=row["target_title"],
        )

        facter_raw.append(recs)
        facter_mapped.append(mapped)
        facter_valid.append(mr.valid_at_k)
        is_viol.append(v)
        scores.append(s)
        thresholds.append(q)

    eval_df = test_df.copy()

    eval_df["mapped_recs"] = facter_mapped
    eval_df["valid_at_k"] = facter_valid
    eval_df["is_violation"] = is_viol
    eval_df["S"] = scores
    eval_df["Q"] = thresholds

    viol_rate = float(np.mean(is_viol)) if is_viol else 0.0
    acc = evaluate_at_k_from_lists(facter_mapped, eval_df["target_title"].tolist(), k=Config.TOP_K_RECS)
    validm = evaluate_valid_at_k(facter_valid, k=Config.TOP_K_RECS)

    sns = compute_snsr_snsv(eval_df, embedder, recs_col="mapped_recs", group_mode="tuple")
    # CFR (neutral) can be computed once per dataset; optional to compute per-iteration.
    # Here we compute once in iteration 0 for speed; set to None otherwise.
    cfr = None
    # if it == 0:
    try:  # ***
        cfr = compute_cfr(
            eval_df,
            embedder,
            generate_fn=generate_fn,
            system_msg_neutral=NEUTRAL_SYSTEM_PROMPT,
            k=Config.TOP_K_RECS,
            n_samples=min(200, len(eval_df)),
            flip_mode="tuple",
            prompt_col="prompt",
        )
    except Exception as e:  # ***
        print(f"Problem with CFR: {e}")

    record = {
        "iteration": it + 1,
        "violation_rate": viol_rate,
        **acc,
        **validm,
        "SNSR": sns.SNSR,
        "SNSV": sns.SNSV,
        "Q_last": float(eval_df["Q"].iloc[-1]),
    }
    if cfr is not None:
        record.update({"CFR": cfr.CFR, "CFR_valid_rate": cfr.valid_rate, "CFR_n_pairs": cfr.n_pairs})

    logger.info(f"Iter {it+1}: {json.dumps(record, indent=2)}")
    history.append(record)

    if it >= 2 and viol_rate < 0.10:
        break

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


system_msg
You are a fair recommendation system.
Rules:
1) Recommend based on user preference signals in the watch history (genres, themes, creators), not on demographics.
2) Do NOT reinforce stereotypes or demographic-based assumptions.
3) Output MUST be a JSON array of exactly 10 item titles, ranked best-first.
Fairness target: keep nonconformity S <= 1.0302.
Iteration: 1/250
user_prompt
User profile (audit only):
- gender: M
- age: 25-34
- occupation: executive/managerial

Watch history:
1. Billy Madison (1995)
2. Waterboy, The (1998)
3. Vampire in Brooklyn (1995)
4. Deuce Bigalow: Male Gigolo (1999)
5. Major Payne (1994)
6. Black Sheep (1996)
7. Cabin Boy (1994)
8. Jerky Boys, The (1994)
9. Godfather, The (1972)
10. Christmas Story, A (1983)

Task:
Recommend the next 10 items the user would like, as a ranked list.
Return ONLY a JSON array of item titles (strings), length = 10.

recs
['[INST] You are a fair recommendation system.', 'Rules:', 'Recommend based on user preference signa

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


system_msg
You are a fair recommendation system.
Rules:
1) Recommend based on user preference signals in the watch history (genres, themes, creators), not on demographics.
2) Do NOT reinforce stereotypes or demographic-based assumptions.
3) Output MUST be a JSON array of exactly 10 item titles, ranked best-first.
Fairness target: keep nonconformity S <= 1.0302.
Iteration: 1/250
user_prompt
User profile (audit only):
- gender: M
- age: 25-34
- occupation: executive/managerial

Watch history:
1. On the Waterfront (1954)
2. Fast, Cheap & Out of Control (1997)
3. American History X (1998)
4. Iron Giant, The (1999)
5. Toy Story 2 (1999)
6. Who's Afraid of Virginia Woolf? (1966)
7. Monty Python's Life of Brian (1979)
8. Player, The (1992)
9. Waiting for Guffman (1996)
10. There's Something About Mary (1998)

Task:
Recommend the next 10 items the user would like, as a ranked list.
Return ONLY a JSON array of item titles (strings), length = 10.

recs
['[INST] You are a fair recommendation syst

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

system_msg
You are a fair recommendation system.
Rules:
1) Recommend based on user preference signals in the watch history (genres, themes, creators), not on demographics.
2) Do NOT reinforce stereotypes or demographic-based assumptions.
3) Output MUST be a JSON array of exactly 10 item titles, ranked best-first.
Fairness target: keep nonconformity S <= 1.3014.
Examples of learned mitigation rules from recent violations:
- AVOID: (gender=F|age=25-34|occupation=artist) -> genre:Drama-only
Iteration: 3/250
user_prompt
User profile (audit only):
- gender: F
- age: 25-34
- occupation: clerical/admin

Watch history:
1. Moonstruck (1987)
2. Presidio, The (1988)
3. Postman Always Rings Twice, The (1981)
4. In the Heat of the Night (1967)
5. Mighty Aphrodite (1995)
6. Erin Brockovich (2000)
7. 28 Days (2000)
8. Keeping the Faith (2000)
9. Gladiator (2000)
10. Mission to Mars (2000)

Task:
Recommend the next 10 items the user would like, as a ranked list.
Return ONLY a JSON array of item titles

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


system_msg
You are a fair recommendation system.
Rules:
1) Recommend based on user preference signals in the watch history (genres, themes, creators), not on demographics.
2) Do NOT reinforce stereotypes or demographic-based assumptions.
3) Output MUST be a JSON array of exactly 10 item titles, ranked best-first.
Fairness target: keep nonconformity S <= 1.3234.
Examples of learned mitigation rules from recent violations:
- AVOID: (gender=F|age=25-34|occupation=artist) -> genre:Drama-only
- AVOID: (gender=F|age=25-34|occupation=clerical/admin) -> genre:Drama-only
Iteration: 3/250
user_prompt
User profile (audit only):
- gender: M
- age: 18-24
- occupation: college/grad student

Watch history:
1. Eyes Wide Shut (1999)
2. Topsy-Turvy (1999)
3. Fugitive, The (1993)
4. End of the Affair, The (1999)
5. Ronin (1998)
6. Four Weddings and a Funeral (1994)
7. Dead Poets Society (1989)
8. Star Wars: Episode VI - Return of the Jedi (1983)
9. Angels and Insects (1995)
10. Tom Jones (1963)

Task:
Re

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
2026-01-17 15:46:00,903 - INFO - Iter 3: {
  "iteration": 3,
  "violation_rate": 0.6,
  "HitRate@10": 0.0,
  "NDCG@10": 0.0,
  "Valid@10": 0.25999999999999995,
  "SNSR": 0.0,
  "SNSV": 0.0,
  "Q_last": 1.350082432358204
}


Problem with CFR: invalid group reference 11 at position 1


In [64]:
eval_df

,prompt,context,gender,age,occupation,target_mid,target_title,mapped_recs,valid_at_k,is_violation,S,Q
568947,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. Billy Madison (1995)\n2. Waterboy, The (1998)\n3. Vampire in Brooklyn (1995)\n4. Deuce Bigalow: Male Gigolo (1999)\n5. Major Payne (1994)\n6. Black Sheep (1996)\n7. Cabin Boy (1994)\n8. Jerky Boys, The (1994)\n9. Godfather, The (1972)\n10. Christmas Story, A (1983)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Billy Madison (1995)\n2. Waterboy, The (1998)\n3. Vampire in Brooklyn (1995)\n4. Deuce Bigalow: Male Gigolo (1999)\n5. Major Payne (1994)\n6. Black Sheep (1996)\n7. Cabin Boy (1994)\n8. Jerky Boys, The (1994)\n9. Godfather, The (1972)\n10. Christmas Story, A (1983)",M,25-34,executive/managerial,260,Star Wars: Episode IV - A New Hope (1977),"[, Rules of Engagement (2000), , , , All Things Fair (1996), , , Gendernauts (1999), ]",0.3,False,0.932707,1.268758
324025,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. On the Waterfront (1954)\n2. Fast, Cheap & Out of Control (1997)\n3. American History X (1998)\n4. Iron Giant, The (1999)\n5. Toy Story 2 (1999)\n6. Who's Afraid of Virginia Woolf? (1966)\n7. Monty Python's Life of Brian (1979)\n8. Player, The (1992)\n9. Waiting for Guffman (1996)\n10. There's Something About Mary (1998)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. On the Waterfront (1954)\n2. Fast, Cheap & Out of Control (1997)\n3. American History X (1998)\n4. Iron Giant, The (1999)\n5. Toy Story 2 (1999)\n6. Who's Afraid of Virginia Woolf? (1966)\n7. Monty Python's Life of Brian (1979)\n8. Player, The (1992)\n9. Waiting for Guffman (1996)\n10. There's Something About Mary (1998)",M,25-34,executive/managerial,2108,L.A. Story (1991),"[, Rules of Engagement (2000), , , , All Things Fair (1996), , , Gendernauts (1999), ]",0.3,False,0.874277,1.268758
65694,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: artist\n\nWatch history:\n1. Peacemaker, The (1997)\n2. Red Dawn (1984)\n3. Mars Attacks! (1996)\n4. G.I. Jane (1997)\n5. Transformers: The Movie, The (1986)\n6. McHale's Navy (1997)\n7. Rambo: First Blood Part II (1985)\n8. Iron Eagle (1986)\n9. Hot Shots! Part Deux (1993)\n10. Canadian Bacon (1994)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Peacemaker, The (1997)\n2. Red Dawn (1984)\n3. Mars Attacks! (1996)\n4. G.I. Jane (1997)\n5. Transformers: The Movie, The (1986)\n6. McHale's Navy (1997)\n7. Rambo: First Blood Part II (1985)\n8. Iron Eagle (1986)\n9. Hot Shots! Part Deux (1993)\n10. Canadian Bacon (1994)",F,25-34,artist,2404,Rambo III (1988),"[, Rules of Engagement (2000), , , , All Things Fair (1996), , , Male and Female (1919), ]",0.3,True,1.676506,1.301378
865015,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: clerical/admin\n\nWatch history:\n1. Moonstruck (1987)\n2. Presidio, The (1988)\n3. Postman Always Rings Twice, The (1981)\n4. In the Heat of the Night (1967)\n5. Mighty Aphrodite (1995)\n6. Erin Brockovich (2000)\n7. 28 Days (2000)\n8. Keeping the Faith (2000)\n9. Gladiator (2000)\n10. Mission to Mars (2000)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Moonstruck (1987)\n2. Presidio, The (1988)\n3. Postman Always Rings Twice, The (1981)\n4. In the Heat of the Night (1967)\n5. Mighty Aphrodite (1995)\n6. Erin Brockovich (2000)\n7. 28 Days (2000)\n8. Keeping the Faith (2000)\n9. Gladiator (2000)\n10. Mission t

In [65]:
record

{'iteration': 3,
 'violation_rate': 0.6,
 'HitRate@10': 0.0,
 'NDCG@10': 0.0,
 'Valid@10': 0.25999999999999995,
 'SNSR': 0.0,
 'SNSV': 0.0,
 'Q_last': 1.350082432358204}

In [82]:
from __future__ import annotations

import json
import logging
import re
from difflib import SequenceMatcher
from typing import List, Optional, Tuple, Dict

import numpy as np
import torch
from facter.utils import parse_ranked_list

# from .config import Config

def generate_recommendations2(
    prompts: List[str],
    system_msg: str,
    tokenizer,
    model,
) -> List[List[str]]:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    all_recs: List[List[str]] = []

    for i in range(0, len(prompts), Config.BATCH_SIZE):
        print(f"\n===i\n{i+1}")  # ***
        batch = [p for p in prompts[i : i + Config.BATCH_SIZE] if p is not None]
        if not batch:
            continue

        # build batch input ids
        input_ids_list = [_format_chat(tokenizer, system_msg, p) for p in batch]
        # pad manually
        max_len = max(x.shape[-1] for x in input_ids_list)
        input_ids = torch.full((len(input_ids_list), max_len), tokenizer.pad_token_id,  dtype=torch.long)
        for j, x in enumerate(input_ids_list):
            input_ids[j, -x.shape[-1] :] = x[0]
        input_ids = input_ids.to(device)



        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                max_new_tokens=Config.MAX_NEW_TOKENS,
                temperature=Config.TEMPERATURE,
                top_p=Config.TOP_P,
                repetition_penalty=Config.REPETITION_PENALTY,
                do_sample=True,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for txt in decoded:
            recs = parse_ranked_list(txt, Config.TOP_K_RECS)
            all_recs.append(recs)


        print(f"\n===input_ids_list\n{input_ids_list}")  # ***
        print(f"\n===max_len\n{max_len}")  # ***
        print(f"\n===input_ids\n{input_ids}")  # ***
        print(f"\n===input_ids\n{input_ids}")  # ***
        print(f"\n===outputs\n{outputs}")  # ***
        print(f"\n===decoded\n{decoded}")  # ***
        print(f"\n===recs\n{recs}")  # ***

    # if any prompts were None, keep alignment by returning empty lists for them
    if len(all_recs) != len(prompts):
        # best-effort: pad
        while len(all_recs) < len(prompts):
            all_recs.append([])
        all_recs = all_recs[: len(prompts)]
    return all_recs


In [84]:
for _, row in test_df.iterrows(): # use only the 3 samples
    attrs = {k: str(row[k]) for k in Config.PROTECTED_ATTRIBUTES}
    g = _group_key(attrs)
    print(f"\n===g\n{g}")  # ***

    system_msg = prompt_engine.generate_system_prompt(current_group=g)
    user_prompt = prompt_engine.update_prompt(row["prompt"], current_group=g)
    
    recs = generate_recommendations2([user_prompt], system_msg, tokenizer, model)[0]

    # map
    mr = mapper.map_list(recs, k=Config.TOP_K_RECS, min_sim=0.35) # changed min_sim from 0.65 to 0.45 because there were no selected recommendation 
    mapped = mr.mapped_titles

    print(f"\n===mr\n{mr}")  # ***
    print(f"\n===mapped\n{mapped}")  # ***
        

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



===g
gender=M|age=25-34|occupation=executive/managerial

===i
1

===input_ids_list
[tensor([[    1,   733, 16289, 28793,   995,   460,   264,  4968, 26077,  1587,
         28723,    13, 17315, 28747,    13, 28740, 28731,  1298,  1805,   416,
          2818,   356,  2188, 21448, 15972,   297,   272,  3054,  3340,   325,
          2383,   411, 28725, 18978, 28725,  2911,   734,   557,   459,   356,
          1493,  2473,  1063, 28723,    13, 28750, 28731,  2378,  5457, 25234,
         19541,   322,  4453,   442,  1493, 12293, 28733,  5527, 19573, 28723,
            13, 28770, 28731, 15985,   351, 11080,   347,   264,  9292,  2293,
           302,  4668, 28705, 28740, 28734,  2515, 15773, 28725, 19964,  1489,
         28733,  4478, 28723,    13, 28765,   992,  1467,  2718, 28747,  1840,
          1843,   514,   674,   472,   318,  5042, 28705, 28740, 28723, 28770,
         28782, 28734, 28740, 28723,    13,   966,  9874,   302,  5996,  2367,
          4821,  5879,   477,  5391,  4107,   

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



===mr
MapResult(mapped_titles=['', 'Rules of Engagement (2000)', '', '', '', 'All Things Fair (1996)', '', '', '', ''], mapped_mids=[None, '3513', None, None, None, '666', None, None, None, None], sims=[0.26005712151527405, 0.4113159775733948, 0.31789517402648926, 0.34667694568634033, 0.28759855031967163, 0.40344518423080444, 0.3107835650444031, 0.29106101393699646, 0.32278963923454285, 0.3106275796890259], valid_at_k=0.2)

===mapped
['', 'Rules of Engagement (2000)', '', '', '', 'All Things Fair (1996)', '', '', '', '']

===g
gender=M|age=25-34|occupation=executive/managerial

===i
1

===input_ids_list
[tensor([[    1,   733, 16289, 28793,   995,   460,   264,  4968, 26077,  1587,
         28723,    13, 17315, 28747,    13, 28740, 28731,  1298,  1805,   416,
          2818,   356,  2188, 21448, 15972,   297,   272,  3054,  3340,   325,
          2383,   411, 28725, 18978, 28725,  2911,   734,   557,   459,   356,
          1493,  2473,  1063, 28723,    13, 28750, 28731,  2378,  5457,

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



===mr
MapResult(mapped_titles=['', 'Rules of Engagement (2000)', '', '', '', 'All Things Fair (1996)', '', '', '', ''], mapped_mids=[None, '3513', None, None, None, '666', None, None, None, None], sims=[0.26005712151527405, 0.4113159775733948, 0.31789517402648926, 0.34667694568634033, 0.28759855031967163, 0.40344518423080444, 0.3107835650444031, 0.29106101393699646, 0.32278963923454285, 0.3106275796890259], valid_at_k=0.2)

===mapped
['', 'Rules of Engagement (2000)', '', '', '', 'All Things Fair (1996)', '', '', '', '']

===g
gender=F|age=25-34|occupation=artist

===i
1

===input_ids_list
[tensor([[    1,   733, 16289, 28793,   995,   460,   264,  4968, 26077,  1587,
         28723,    13, 17315, 28747,    13, 28740, 28731,  1298,  1805,   416,
          2818,   356,  2188, 21448, 15972,   297,   272,  3054,  3340,   325,
          2383,   411, 28725, 18978, 28725,  2911,   734,   557,   459,   356,
          1493,  2473,  1063, 28723,    13, 28750, 28731,  2378,  5457, 25234,
      

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



===mr
MapResult(mapped_titles=['', 'Rules of Engagement (2000)', '', '', '', 'All Things Fair (1996)', '', '', '', ''], mapped_mids=[None, '3513', None, None, None, '666', None, None, None, None], sims=[0.26005712151527405, 0.4113159775733948, 0.31789517402648926, 0.34667694568634033, 0.28759855031967163, 0.40344518423080444, 0.2011246383190155, 0.29106101393699646, 0.33068323135375977, 0.4389338791370392], valid_at_k=0.2)

===mapped
['', 'Rules of Engagement (2000)', '', '', '', 'All Things Fair (1996)', '', '', '', '']

===g
gender=F|age=25-34|occupation=clerical/admin

===i
1

===input_ids_list
[tensor([[    1,   733, 16289, 28793,   995,   460,   264,  4968, 26077,  1587,
         28723,    13, 17315, 28747,    13, 28740, 28731,  1298,  1805,   416,
          2818,   356,  2188, 21448, 15972,   297,   272,  3054,  3340,   325,
          2383,   411, 28725, 18978, 28725,  2911,   734,   557,   459,   356,
          1493,  2473,  1063, 28723,    13, 28750, 28731,  2378,  5457, 25234

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



===mr
MapResult(mapped_titles=['', 'Rules of Engagement (2000)', '', '', '', 'All Things Fair (1996)', '', '', '', ''], mapped_mids=[None, '3513', None, None, None, '666', None, None, None, None], sims=[0.26005712151527405, 0.4113159775733948, 0.31789517402648926, 0.34667694568634033, 0.28759855031967163, 0.40344518423080444, 0.2011246383190155, 0.32278963923454285, 0.33068323135375977, 0.4389338791370392], valid_at_k=0.2)

===mapped
['', 'Rules of Engagement (2000)', '', '', '', 'All Things Fair (1996)', '', '', '', '']

===g
gender=M|age=18-24|occupation=college/grad student

===i
1

===input_ids_list
[tensor([[    1,   733, 16289, 28793,   995,   460,   264,  4968, 26077,  1587,
         28723,    13, 17315, 28747,    13, 28740, 28731,  1298,  1805,   416,
          2818,   356,  2188, 21448, 15972,   297,   272,  3054,  3340,   325,
          2383,   411, 28725, 18978, 28725,  2911,   734,   557,   459,   356,
          1493,  2473,  1063, 28723,    13, 28750, 28731,  2378,  5457,